실습 6. 주기별 복합 데이터 정제
- 여러 정규화 주기로 정제해 행·결측·보간 결과 비교

목표
- 여러 정규화 주기로 정제해 행 수·결측·보간 결과를 비교

단계
- 10초·30초·1분 주기로 각각 격자 정규화
- 각 주기의 행 수와 정규화 직후 결측을 세기
- 시간 보간 후 결측이 모두 사라지는지 확인

예상 결과
- 주기가 길수록 행·결측이 줄고, 보간 후 모두 0

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_theme(style="whitegrid")

# 한글깨짐 해결
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

df = pd.read_csv('../data/22_열처리.csv', encoding='utf-8')

df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.set_index('timestamp').sort_index()
norm = df[['제어출력', '소입로온도']].asfreq('10s')

In [ ]:
# [정규화 타겟 주기(10초, 30초, 1분)별 누락 패턴과 보간 동작 자동 대조]
# 1. '10s', '30s', '1min' 등 다운샘플링의 강도가 다른 주기를 순회하며 각각 `asfreq` 격자 크기 변화를 봅니다.
# 2. 주기가 길어질수록(1분) 총 행의 수와 그로 인해 채워야 할 NaN(결측)의 분량이 변화하며 보간 후 결과는 모두 0으로 완수됩니다.
# * 분석가에게 고해상도(10s) 가공을 쓸지, 장기 상태 판독을 위해 가벼운 저해상도(1min) 가공을 채택할지
#   주기별 보간 통계치를 비교해 적정 분석 주기를 조절할 수 있는 유연성을 제공합니다.
# * 루프 내부에서 `freq` 인자를 문자열 형태로 `asfreq()`에 적확하게 전달해야 격자가 맞게 구성됩니다.

for freq in ['10s', '30s', '1min']:
    n = df[['제어출력']].asfreq(freq)
    c = n['제어출력'].interpolate(method='time')
    print(freq, '행', len(n), '결측', int(n['제어출력'].isna().sum()), '보간후', int(c.isna().sum()))
# 10s 행 200 결측 22 보간후 0
# 30s 행 67 결측 9 보간후 0
# 1min 행 34 결측 5 보간후 0

10s 행 200 결측 22 보간후 0
30s 행 67 결측 9 보간후 0
1min 행 34 결측 5 보간후 0
